<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_94_conditional_image_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🖼️ **Module 94 — Conditional Image Translation** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 13 · Vision Robustness & Generative Translation


# Module 94 — Conditional Image Translation

> M85 built **unconditional** generative models (GAN · AE · VAE). This module is the **conditional** sibling — *"input is an image; output is the same image translated into a different domain."* sketch → photo, day → night, horse → zebra, anime → realistic, face → face-of-different-age. Built on ndb796's `Pix2Pix_for_Facades` and `StarGAN_Tutorial` notebooks, this module re-derives the **Pix2Pix** (paired) · **CycleGAN** (unpaired) · **StarGAN-v1/v2** (multi-domain) trio from scratch, then walks through the diffusion-era successors that quietly took over by 2024 (**ControlNet · InstructPix2Pix · IP-Adapter · ZeroShot-I2I · Stable Cascade**).

### What you'll cover
1. The taxonomy — **paired · unpaired · multi-domain · zero-shot**
2. **Pix2Pix** (Isola 2017) — the conditional GAN + L1 + PatchGAN recipe
3. **U-Net generator** + **PatchGAN discriminator** under the hood
4. **CycleGAN** (Zhu 2017) — *unpaired* translation via cycle consistency
5. **CycleGAN failure modes** + identity loss + perceptual augmentations
6. **StarGAN-v1 / v2** (Choi 2018, 2020) — one model, many domains
7. **Image-to-image with attention** — SPADE, SEAN, MUNIT, DRIT, UVCGAN
8. **The diffusion takeover** — ControlNet · InstructPix2Pix · IP-Adapter
9. **Evaluation** — FID · KID · LPIPS · IS · SegA · user studies
10. **The 2026 picker** — when to use which method


## 1 · The taxonomy

| Paradigm | Data assumption | Canonical paper |
|---|---|---|
| **Paired** | Aligned `(x_A, x_B)` pairs (same scene in both domains) | Pix2Pix (Isola 2017) |
| **Unpaired** | Two unaligned sets `X_A` and `X_B`; no correspondence | CycleGAN (Zhu 2017) · UNIT · DiscoGAN · DualGAN |
| **Multi-domain** | One model serving N domains via a domain label | StarGAN-v1/v2 (Choi 2018, 2020) |
| **Conditional + multi-modal** | One input image → many plausible outputs | BicycleGAN · MUNIT · DRIT · multi-modal CycleGAN |
| **Zero-shot / text-conditioned** | Translation via natural-language instruction | InstructPix2Pix · ControlNet · IP-Adapter (M86) |

**Why the field bothered with each.** Paired data is rare (you can't take a photo of the same horse as a zebra). Unpaired translation unblocks the long tail (style transfer, season change, photo enhancement). Multi-domain reduces N(N−1) Pix2Pix models to 1. Text-conditioned makes the translation language-controllable. Each step expands the data regime that's economically viable.

> 📍 **A useful mental anchor.** All conditional image translation = "supervised regression on an image-to-image map" + "discriminator to force the output onto the target distribution" + "a domain constraint that makes the problem well-posed." Different methods make different choices about each of the three.


## 2 · Pix2Pix — paired with conditional GAN

**Pix2Pix** (Isola, Zhu, Zhou, Efros, CVPR 2017): given paired data `(x, y)` (e.g. building facade segmentation → photo), train a conditional GAN.

```
Generator G(x):       sketch  → photo
Discriminator D(x,y): (sketch, photo) → real / fake
                                 ▲
                  conditioned on the input too!
```

The generator is a **U-Net** (encoder-decoder with skip connections); the discriminator is a **PatchGAN** (judges 70×70 patches, not the whole image — more stable training, sharper outputs).

**Loss:**
```
L_GAN(G, D) = E[log D(x,y)] + E[log(1 − D(x, G(x)))]
L_L1(G)     = E[||y − G(x)||₁]
L_total     = L_GAN(G, D) + λ · L_L1(G)        # λ = 100
```

Why **L1** rather than L2? L1 produces sharper images (L2 averages over plausible outputs → blur). The discriminator forces high-frequency detail; L1 forces low-frequency correctness. **Same network handles edge → photo, segmentation → photo, day → night, B&W → color, sketch → bag, satellite → map, facade → photo.**

> 🔁 **One generator, eight applications.** The original paper showed Pix2Pix is **architecture-agnostic** — change the dataset, get a different translation. This was the moment image translation became a *recipe* instead of a per-domain problem.


## 3 · U-Net generator + PatchGAN discriminator

**U-Net** (Ronneberger 2015, originally for biomedical segmentation): symmetric encoder-decoder with **skip connections** at each spatial resolution.

```
                 (skip)
input 256² ─►Enc1─►Enc2─►Enc3─►Enc4─►Bottleneck─►Dec4─►Dec3─►Dec2─►Dec1─►output 256²
              │     │     │     │                  ▲     ▲     ▲     ▲
              └─────────────────┴──────────────────┘─────┘─────┘─────┘
```

The skip connections carry low-level signal (edges, color regions) past the bottleneck, so the decoder doesn't have to re-invent them from a compressed latent. **Why every image-to-image net since uses U-Net or a U-Net variant.** Stable Diffusion's U-Net (M86) is the same idea scaled up with attention.

**PatchGAN** (Isola 2017): the discriminator is **fully convolutional**, output is a `30×30` grid of "real/fake" decisions — each cell judges a `70×70` patch of the input. Mean over the grid = total loss.

| Receptive field | Effect |
|---|---|
| 1×1 (pixel) | Color matching only, no texture |
| 16×16 | Local texture, no global structure |
| **70×70** (default) | **Sharp local texture + reasonable global** |
| 286×286 (whole image) | Slower, harder to train, marginal quality gain |

PatchGAN is **lighter** (fewer parameters than a full-image discriminator) and **more stable** (averaging many local decisions). Used by Pix2Pix, CycleGAN, StarGAN, BigGAN-deep, StyleGAN-2 in some configurations.


In [ ]:
import torch, torch.nn as nn

class UNetBlock(nn.Module):
    def __init__(self, ic, oc, down=True, drop=False):
        super().__init__()
        if down:
            self.conv = nn.Conv2d(ic, oc, 4, 2, 1)
        else:
            self.conv = nn.ConvTranspose2d(ic, oc, 4, 2, 1)
        self.norm = nn.BatchNorm2d(oc)
        self.act  = nn.LeakyReLU(0.2) if down else nn.ReLU()
        self.drop = nn.Dropout(0.5) if drop else nn.Identity()
    def forward(self, x):
        return self.drop(self.act(self.norm(self.conv(x))))

class PatchGAN(nn.Module):
    def __init__(self, ic=6):                # in: concat(x, y) → 3+3 = 6
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 1, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1,   4, 1, 1),  # 30×30 patch grid for 256² input
        )
    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))


## 4 · CycleGAN — unpaired translation via cycle consistency

Pix2Pix needs aligned pairs. **CycleGAN** (Zhu, Park, Isola, Efros, ICCV 2017) removes that requirement with one extra constraint: **cycle consistency**.

```
G: X → Y    F: Y → X      (two generators)
D_X, D_Y    (two discriminators)

Forward cycle:   x ─G→ G(x) ─F→ F(G(x))   ≈ x
Backward cycle:  y ─F→ F(y) ─G→ G(F(y))   ≈ y
```

**Loss:**
```
L_GAN(G, D_Y, X, Y) = E[log D_Y(y)] + E[log(1 − D_Y(G(x)))]
L_GAN(F, D_X, Y, X) = symmetric
L_cyc(G, F)         = E[||F(G(x)) − x||₁] + E[||G(F(y)) − y||₁]
L_total             = L_GAN + L_GAN + λ · L_cyc        # λ = 10
```

The cycle loss says **"translating there and back should return the original"** — this enforces *content preservation* without requiring a paired counterpart. The two GAN losses ensure each translated image lands on the target distribution.

**Applications.** Horse ↔ zebra, summer ↔ winter, Monet ↔ photo, apples ↔ oranges, day ↔ night satellite imagery, cycleGAN-medical (CT ↔ MR), CycleGAN voice. All from unaligned data.

> ⚙️ **The "identity loss" trick** (Section 5.2 of the paper). Add `L_id = E[||G(y) − y||₁]` — "if you feed a target-domain image to G, it should be a no-op." Stabilizes color preservation; without it, CycleGAN sometimes inverts colors or hue-shifts the whole image while satisfying cycle consistency.


## 5 · CycleGAN failure modes + fixes

CycleGAN is brilliant but fails in predictable ways. Knowing them keeps you out of trouble:

| Failure | Symptom | Fix |
|---|---|---|
| **Steganography** (Chu 2017) | Generator hides info about `x` in invisible patterns to satisfy cycle | Add noise to intermediate `G(x)` |
| **Shape change refusal** | Horse → zebra with horse shape unchanged ✅ but apple → orange where shape should change ❌ | Use **MUNIT / DRIT** with explicit shape vs style codes |
| **Hallucinated content** | Photo of empty road → zebra crossing painted in | Add **perceptual / segmentation-aware loss** |
| **Mode collapse** | All zebras look identical | Multi-scale discriminator + L1 noise injection |
| **Color drift** | Subtle hue shift across domain | **Identity loss** + match histogram in latent space |
| **Texture artifacts** | Repeating "GAN crosshatch" pattern | LPIPS loss + spectral norm + R1 regularizer |

A useful sanity test: compare `F(G(x))` to `x` pixel-by-pixel. Cycle loss can be `0.001` while the model has memorized invisible watermarks. The visible-similarity should match the cycle-loss number.

> 🧪 **CycleGAN diagnostics.** `lpips(F(G(x)), x) < 0.05` is the modern proxy for *true* cycle consistency, much harder to fake than L1.


## 6 · StarGAN-v1 + v2 — one model, many domains

**Problem.** If you have **N** domains (different hair colors, ages, expressions), Pix2Pix needs N(N−1) models, CycleGAN needs N(N−1)/2 pairs. **StarGAN-v1** (Choi 2018) reduces all of it to **one** model.

```
Generator G(x, c_target):  image + target-domain label → translated image
Discriminator D(x):
    D_real_fake(x):   real/fake
    D_domain(x):      which domain does this image belong to?
```

The generator takes a one-hot domain code; the discriminator has two heads — real/fake + domain classification. Loss is the obvious extension of cycle GAN + a **domain classification loss** so `G(x, c)` reliably lands in domain `c`.

**Result.** One model handles every (source, target) pair across all N domains in a single forward. Trained on CelebA with `{black hair, blond hair, brown hair, male, young}` — five binary axes → `2^5 = 32` target combinations.

**StarGAN-v2** (Choi 2020) replaces the one-hot code with a **style code** sampled from a domain-specific mapping network — the model now produces *diverse* outputs within each target domain (e.g. multiple plausible blond hairstyles, not one canonical blond). Architecture: AdaIN-style modulation (M92!) of the generator backbone using the style code. **Achieves SOTA on CelebA-HQ and AFHQ until the diffusion era.**

> 🎯 **A note on practical generality.** StarGAN-v2 is a *very* good general image-to-image net for 2020-2022 multi-domain problems. By 2026 it's mostly replaced by ControlNet / IP-Adapter but the **mapping-network + AdaIN-modulated generator** architecture lives on in StyleGAN3, Stable Cascade, FLUX.1-dev — the lineage is direct.


## 7 · Image-to-image with attention and disentanglement

A second wave of papers improved on Pix2Pix / CycleGAN with three orthogonal axes:

| Method | What it adds | When useful |
|---|---|---|
| **SPADE / GauGAN** (Park 2019) | Per-pixel modulation from a semantic map; AdaIN on segmentation | Semantic-map → photo (the NVIDIA "AI Painting" demo) |
| **SEAN** (Zhu 2020) | Per-region style codes on top of SPADE | Editable photoreal portraits |
| **MUNIT** (Huang 2018) | Content + style decomposition; **multi-modal** outputs | When N plausible outputs are valid |
| **DRIT++** (Lee 2020) | Cross-domain disentangled representation | Style-only or content-only swap |
| **UVCGAN** (Torbunov 2022) | Vision Transformer backbone for CycleGAN | When you need high-frequency detail |
| **U-GAT-IT** (Kim 2020) | Attention + adaptive layer-instance norm | Selfie ↔ anime (the production version) |
| **CUT / FastCUT** (Park 2020) | Replace cycle loss with patch-NCE contrastive | Faster, often higher-quality, no second generator |

**CUT in one line** changes the loss landscape: contrast each translated patch against its source-image counterpart (positive) and other patches (negatives). Train 50% faster than CycleGAN, often higher LPIPS quality.

```
L_NCE = −log [exp(s(f(x), f(G(x)))) / Σ_neg exp(s(f(x), f(G(x))_neg))]
```

CUT and U-GAT-IT are the **2022-vintage state of the art** for unpaired translation before diffusion completely took over.


## 8 · The diffusion takeover (2022 → 2026)

By 2022 a quiet shift happened: diffusion models (M21, M86) became better at image translation than purpose-built GANs.

| Method | Idea | Best for |
|---|---|---|
| **SDEdit** (Meng 2022) | Add noise to source image, denoise with target-domain SD | Quick stylization with `0.4-0.8` noise level |
| **InstructPix2Pix** (Brooks 2023) | Train SD on `(image, instruction, edited image)` triples from GPT-3 + Stable Diffusion | Text-instructed edits ("turn the dog into a cat") |
| **ControlNet** (Zhang 2023) | A side-network conditioned on canny / depth / pose / segmentation | Preserves spatial layout while replacing style/content |
| **IP-Adapter** (Ye 2023) | CLIP-image embedding via cross-attention | One-shot style transfer with no fine-tune |
| **Null-text inversion / Prompt-to-Prompt** (Hertz 2023) | Invert image → SD latent, then edit prompt | Surgical text-based edits |
| **DiffEdit** (Couairon 2022) | Mask-conditioned editing from a text instruction | Object-only edits |
| **MasaCtrl** (Cao 2023) | Mutual self-attention control across diffusion steps | Consistent character editing |
| **PnP-Diffusion** (Tumanyan 2023) | Plug-and-play feature injection during sampling | Style swap that preserves structure |

**Why diffusion won.** Three reasons: (a) the prior (a frontier text-to-image SD/FLUX) already knows what natural images look like — no need to learn it from scratch on a 10k-image domain set; (b) **text-controllability** makes the translation user-friendly; (c) **iterative denoising** provides natural intermediate sampling — easier to control than a single forward.

> 🔄 **A subtle but important point.** The 2024-2026 "best image-to-image" model isn't a specialized translator — it's a frontier text-to-image diffusion model + ControlNet + IP-Adapter + an `image-to-image` pipeline (`strength=0.6`). The architecture from M86 *is* the image translator. Pix2Pix / CycleGAN / StarGAN are now mostly pedagogical and used in resource-constrained domains (medical, scientific imaging) where pretraining on natural images doesn't transfer.


## 9 · Evaluation — what to measure

Image translation is hard to evaluate because **there is no single correct output**. Six common metrics:

| Metric | Measures | Best when |
|---|---|---|
| **FID** (Heusel 2017) | Inception feature distribution distance source vs target | The target distribution is known (you have ground-truth set) |
| **KID** (Bińkowski 2018) | Same idea, unbiased + works at small N | When sample sizes are small (<10k) |
| **LPIPS** (Zhang 2018) | Perceptual distance between input + output | Paired data — measure content preservation |
| **IS** (Inception Score) | Diversity × confidence | Old metric, often deceptive — prefer FID/KID |
| **SegA / SemA** | Re-segment output, check pixel-class match | Map / segmentation translation |
| **User study (MOS, AB)** | Human comparison | Final tie-breaker for production claims |

**A practical eval template (2026):**
1. FID / KID for distribution match
2. LPIPS to source for content preservation
3. CLIP-image-image similarity to a "good" reference
4. 5-person blind AB test on `n=20` outputs for final sanity

**Anti-pattern**: reporting only FID. FID can be gamed by overfitting to the eval set's Inception features; combine with at least one perceptual metric and one human eval.

> 📏 **The 2026 metric soup.** With diffusion models, eval has grown a CLIP-based zoo: **CLIPScore** (image-text alignment), **DINO-FID** (FID with DINO-v2 features instead of Inception), **HPS** (Human-Preference Score), **PickScore**, **Aesthetic Predictor v2.5**. For image-to-image tasks the most-reported are **FID + LPIPS + CLIP-image-similarity**.


## 10 · The 2026 picker

```
Have aligned (x_A, x_B) pairs?
├── YES → Pix2Pix    (small data) | InstructPix2Pix (large data + text)
└── NO  → multi-domain (N>2) ?
           ├── YES → StarGAN-v2  (specialized)  | IP-Adapter + SDXL (general)
           └── NO  → unpaired (X_A, X_B) sets ?
                    ├── YES (small data, niche domain like medical) → CycleGAN / CUT
                    └── YES (general images, plenty of compute)   → SDEdit + ControlNet
```

**By problem class:**

| Problem | 2026 default |
|---|---|
| Sketch → photo | InstructPix2Pix or Pix2Pix-large + ControlNet-Scribble |
| Day → night | SDEdit(strength=0.6) + prompt "at night" + ControlNet-Depth |
| Face attribute edit (hair color, age) | StarGAN-v2 (cheap), InstantID + IP-Adapter (best quality) |
| Style transfer (any → Van Gogh) | InstantStyle / IP-Adapter (M92) |
| Map → satellite | Pix2Pix + ControlNet-Tile (high-res preserved) |
| Medical image translation (CT → MR) | **CycleGAN / CUT** — domain-specific, regulated, can't use SD prior |
| Anime → real face | U-GAT-IT or StarGAN-v2 (cheap), DDPM-Inversion + identity-LoRA (quality) |
| One-shot personalization | IP-Adapter / InstantID (M86 callbacks) |

> 🎓 **The 2017 → 2026 arc, one sentence.** Image-to-image translation went from "train one CNN per domain pair with millions of labels" → "train one CNN with adversarial loss on unaligned data" → "train one big multi-domain model" → "fine-tune a frontier diffusion prior with ControlNet" → "zero-shot edit with text + image conditioning." Each step traded **more model knowledge for less per-task data**. The intellectual algorithm is the same; the data regime that's economically viable grew enormously.

## ✅ Recap

- **Pix2Pix** = conditional GAN + L1 + U-Net + PatchGAN. Needs paired data. 8 applications, 1 recipe.
- **CycleGAN** = two GANs + cycle consistency for unpaired data. Watch for steganography, shape refusal, identity-loss need.
- **StarGAN-v1/v2** = one model for N domains via domain code (v1) or style code + AdaIN-modulation (v2).
- **SPADE/SEAN/MUNIT/DRIT/CUT/U-GAT-IT** = 2018-2022 improvements: attention, disentanglement, contrastive loss, ViT backbones.
- **2022+ diffusion takeover**: SDEdit, InstructPix2Pix, ControlNet, IP-Adapter — better quality + text controllability.
- **Eval**: FID/KID for distribution, LPIPS for content preservation, CLIP-similarity, human AB.
- **2026 picker**: paired → Pix2Pix/InstructPix2Pix; multi-domain → StarGAN-v2/IP-Adapter; unpaired specialized → CycleGAN; general → SD + ControlNet.

Next: **M95 — Image Captioning + Adversarial NLP** (the final module of Part 13).
